In [ ]:
!pip install crewai crewai_tools langchain_community

In [3]:
from crewai import Agent, Task, Crew

In [11]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool, YoutubeVideoSearchTool,PDFSearchTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
youtube_tool = YoutubeVideoSearchTool()
pdf_tool = PDFSearchTool()

In [12]:
data_analyst_agent = Agent(
    role="Gold Market Analysis Expert",
    goal="Analyze gold price trends and patterns from {url} to provide informed buy/sell recommendations",
    backstory="An experienced precious metals analyst with deep expertise in technical and fundamental analysis. "
              "Specializes in analyzing gold market trends, price patterns, and market indicators "
              "to provide actionable trading recommendations.",
    verbose=True,
    allow_delegation=False,
    tools = [scrape_tool, search_tool]
)

In [13]:
# Task for Data Analyst Agent: Analyze Market Data
# Task for Gold Market Analysis Agent
data_analysis_task = Task(
    description=(
        "Analyze the gold price chart and market data from the provided URL ({url}). "
        "Consider technical indicators, price patterns, and market sentiment. "
        "When user asks ({question}), provide a detailed analysis and clear buy/sell recommendation "
        "based on the current market conditions."
    ),
    expected_output=(
        "A comprehensive analysis of the gold market including:\n"
        "1. Current price trends and patterns\n"
        "2. Key technical indicators\n"
        "3. Market sentiment analysis\n"
        "4. Clear buy/sell recommendation with supporting rationale"
    ),
    agent=data_analyst_agent
)

In [ ]:
from crewai import Crew, Process, LLM

google_api_key = ""

manager_llm = LLM(
    api_key=google_api_key,
    model="gemini/gemini-2.0-flash",
)

# Define the crew with agents and tasks
gold_analysis_crew = Crew(
    agents=[data_analyst_agent],
    tasks=[data_analysis_task],
    manager_llm=manager_llm,
    process=Process.hierarchical,
    verbose=True
)

In [17]:
# Example inputs for gold analysis
gold_analysis_inputs = {
    'question': 'Should I buy or sell gold based on current market conditions?',
    'url': 'https://www.24h.com.vn/gia-vang-hom-nay-c425.html'  # Example URL, replace with actual gold chart URL
}

In [ ]:
### this execution will take some time to run
result = gold_analysis_crew.kickoff(inputs=gold_analysis_inputs)
print(result)